In [18]:
import pandas as pd
from datetime import datetime

In [6]:
# Path to the template TSV file containing the headers
template_file_path = "node_template/submission_demographic_template.tsv"

# Read the template TSV file to extract the headers
df_template = pd.read_csv(template_file_path, sep="\t", nrows=0)  # Read only the header
headers = df_template.columns.tolist()  # Extract the headers as a list

In [7]:
# Observational Patients
# File paths
case_path_obs = "/Users/jinn/Documents/IU/ARDaC/case_obs_DCC_data_release_v2-0-0.tsv"
file_path_obs = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/OBS Final Datasets/OBS_SUBJECTS.csv"

# Read the files using pandas
df_obs_case = pd.read_csv(case_path_obs, sep="\t", dtype=str)
df_obs_input = pd.read_csv(file_path_obs, sep=",", dtype=str)

df_obs_output = pd.DataFrame(index=df_obs_input.index, columns=headers)

In [20]:
# Step 1: Extract "*submitter_id" from df_obs_case and create case_table
case_table = pd.DataFrame()
case_table["*submitter_id"] = df_obs_case["*submitter_id"]
case_table["usubjid"] = case_table["*submitter_id"].apply(lambda x: x.split("_")[0])  # Extract the number before "_"

# Step 2: Iterate through case_table and map values to df_obs_output
for _, row in case_table.iterrows():
    submitter_id = row["*submitter_id"]
    usubjid = row["usubjid"]

    # Find the corresponding record in df_obs_input
    input_row = df_obs_input[df_obs_input["usubjid"] == usubjid]

    if not input_row.empty:
        input_row = input_row.iloc[0]  # Extract the first matching row

        # Populate df_obs_output
        df_obs_output.loc[:, "*type"] = "demographic"
        df_obs_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
        df_obs_output.loc[_, "*submitter_id"] = f"{submitter_id}_demographic"
        df_obs_output.loc[_, "*cases.submitter_id"] = f"{submitter_id}"
        df_obs_output.loc[_, "age_at_index"] = input_row.get("calc_age", None)
        df_obs_output.loc[_, "cause_of_death_primary"] = input_row.get("codp", None)
        df_obs_output.loc[_, "cause_of_death_secondary"] = input_row.get("cods", None)
        df_obs_output.loc[_, "cur_employ_stat"] = input_row.get("employed", None)
        df_obs_output.loc[_, "education"] = input_row.get("edu", None)
        df_obs_output.loc[_, "ethnicity"] = input_row.get("ethnic", None)
        df_obs_output.loc[_, "gender"] = input_row.get("gender", None)
        df_obs_output.loc[_, "marital"] = input_row.get("maristat", None)
        df_obs_output.loc[_, "race"] = input_row.get("race", None)
        df_obs_output.loc[_, "sex"] = input_row.get("sex", None)

        # Map "vital_status" from "ALIVE"
        alive = input_row.get("ALIVE", "").strip()
        if alive == "Y":
            df_obs_output.loc[_, "vital_status"] = "Alive"
        elif alive == "N":
            df_obs_output.loc[_, "vital_status"] = "Dead"
        else:
            df_obs_output.loc[_, "vital_status"] = "Not Reported"

        # Extract "year_of_birth" and "year_of_death"
        brthdtc = input_row.get("brthdtc", None)
        df_obs_output.loc[_, "year_of_birth"] = brthdtc.split("-")[0] if pd.notna(brthdtc) else None

        dthdtc = input_row.get("dthdtc", None)
        df_obs_output.loc[_, "year_of_death"] = dthdtc.split("-")[0] if pd.notna(dthdtc) else None

        # Calculate "days_to_death"
        scdat = input_row.get("scdat", None)  # Study enrollment date
        if pd.notna(dthdtc) and pd.notna(scdat):
            try:
                death_date = datetime.strptime(dthdtc, "%Y-%m-%d")
                study_date = datetime.strptime(scdat, "%Y-%m-%d")
                days_to_death = (death_date - study_date).days
                df_obs_output.loc[_, "days_to_death"] = days_to_death
            except ValueError:
                df_obs_output.loc[_, "days_to_death"] = None
        

In [21]:
df_obs_output

,*type,project_id,*submitter_id,*cases.submitter_id,age_at_index,cause_of_death_primary,cause_of_death_secondary,cur_employ_stat,days_to_death,death_related_to,education,ethnicity,gender,marital,race,sex,vital_status,year_of_birth,year_of_death
0,demographic,ARDaC-AlcHepNet,11001_obs_demographic,11001_obs,57.2,NaN,NaN,No,NaN,NaN,Trade School/Some College,Non-hispanic,Male,Married,White,Male,Alive,1962,None
1,demographic,ARDaC-AlcHepNet,11002_obs_demographic,11002_obs,60.8,NaN,NaN,No,NaN,NaN,Standard College/University,Non-hispanic,Female,Married,Native Hawaiian or Other Pacific Islander,Female,Alive,1958,None
2,demographic,ARDaC-AlcHepNet,11003_obs_demographic,11003_obs,35.9,NaN,NaN,No,NaN,NaN,Standard College/University,Non-hispanic,Male,"Single, never married",White,Male,Alive,1983,None
3,demographic,ARDaC-AlcHepNet,11004_obs_demographic,11004_obs,55,NaN,NaN,Yes,NaN,NaN,Completed High School,Non-hispanic,Male,Married,White,Male,Alive,1964,None
4,demographic,ARDaC-AlcHepNet,11006_obs_demographic,11006_obs,49.4,Septic Shock,Acute kidney injury Lactic acidosis Spontaneou...,No,101,NaN,Trade School/Some College,Non-hispanic,Male,Living with significant other (common law marr...,White,Male,Dead,1970,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,demographic,ARDaC-AlcHepNet,81207_obs_demographic,81207_obs,37.4,NaN,NaN,Yes,NaN,NaN,Trade School/Some College,Non-hispanic,Male,Divorced,Black or African American,Male,Alive,1986,None
1130,demographic,ARDaC-AlcHepNet,81208_obs_demographic,81208_obs,33.2,NaN,NaN,Yes,NaN,NaN,Standard College/University,Non-hispanic,Female,Married,White,Female,Alive,1990,None
1131,demographic,ARDaC-AlcHepNet,81209_obs_demographic,81209_obs,34.8,NaN,NaN,No,NaN,NaN,Trade School/Some College,Non-hispanic,Female,Divorced,White,Female,Alive,1988,None
1132,demographic,ARDaC-AlcHepNet,81210_obs_demographic,81210_obs,55.8,NaN,NaN,No,NaN,NaN,Standard College/University,Non-hispanic,Female,Divorced,White,Female,Alive,1968,None


In [22]:
# Export the "DEMOGRAPHIC" node for Observational study.
obs_output_path = "demographic_obs_DCC_data_release_v2-0-0.tsv"
df_obs_output.to_csv(obs_output_path, sep="\t", index=False, header=True)
print(f"Observational patients file saved as: {obs_output_path}")

Observational patients file saved as: demographic_obs_DCC_data_release_v2-0-0.tsv


In [23]:
# Clinical Trial Patients
# File paths
case_path_rct = "/Users/jinn/Documents/IU/ARDaC/case_rct_DCC_data_release_v2-0-0.tsv"
file_path_rct = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_SUBJECTS.csv"

# Read the files using pandas
df_rct_case = pd.read_csv(case_path_rct, sep="\t", dtype=str)
df_rct_input = pd.read_csv(file_path_rct, sep=",", dtype=str)

df_rct_output = pd.DataFrame(index=df_rct_input.index, columns=headers)

In [24]:
# Step 1: Extract "*submitter_id" from df_rct_case and create case_table
case_table = pd.DataFrame()
case_table["*submitter_id"] = df_rct_case["*submitter_id"]
case_table["usubjid"] = case_table["*submitter_id"].apply(lambda x: x.split("_")[0])  # Extract the number before "_"

# Step 2: Iterate through case_table and map values to df_rct_output
for _, row in case_table.iterrows():
    submitter_id = row["*submitter_id"]
    usubjid = row["usubjid"]

    # Find the corresponding record in df_rct_input
    input_row = df_rct_input[df_rct_input["usubjid"] == usubjid]

    if not input_row.empty:
        input_row = input_row.iloc[0]  # Extract the first matching row

        # Populate df_rct_output
        df_rct_output.loc[:, "*type"] = "demographic"
        df_rct_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
        df_rct_output.loc[_, "*submitter_id"] = f"{submitter_id}_demographic"
        df_rct_output.loc[_, "*cases.submitter_id"] = f"{submitter_id}"
        df_rct_output.loc[_, "age_at_index"] = input_row.get("calc_age", None)
        df_rct_output.loc[_, "cause_of_death_primary"] = input_row.get("codp", None)
        df_rct_output.loc[_, "cause_of_death_secondary"] = input_row.get("cods", None)
        df_rct_output.loc[_, "cur_employ_stat"] = input_row.get("employed", None)
        df_rct_output.loc[_, "education"] = input_row.get("edu", None)
        df_rct_output.loc[_, "ethnicity"] = input_row.get("ethnic", None)
        df_rct_output.loc[_, "gender"] = input_row.get("gender", None)
        df_rct_output.loc[_, "marital"] = input_row.get("maristat", None)
        df_rct_output.loc[_, "race"] = input_row.get("race", None)
        df_rct_output.loc[_, "sex"] = input_row.get("sex", None)

        # Map "vital_status" from "ALIVE"
        alive = input_row.get("ALIVE", "").strip()
        if alive == "Y":
            df_rct_output.loc[_, "vital_status"] = "Alive"
        elif alive == "N":
            df_rct_output.loc[_, "vital_status"] = "Dead"
        else:
            df_rct_output.loc[_, "vital_status"] = "Not Reported"

        # Extract "year_of_birth" and "year_of_death"
        brthdtc = input_row.get("brthdtc", None)
        df_rct_output.loc[_, "year_of_birth"] = brthdtc.split("-")[0] if pd.notna(brthdtc) else None

        dthdtc = input_row.get("dthdtc", None)
        df_rct_output.loc[_, "year_of_death"] = dthdtc.split("-")[0] if pd.notna(dthdtc) else None

        # Calculate "days_to_death"
        scdat = input_row.get("scdat", None)  # Study enrollment date
        if pd.notna(dthdtc) and pd.notna(scdat):
            try:
                death_date = datetime.strptime(dthdtc, "%Y-%m-%d")
                study_date = datetime.strptime(scdat, "%Y-%m-%d")
                days_to_death = (death_date - study_date).days
                df_rct_output.loc[_, "days_to_death"] = days_to_death
            except ValueError:
                df_rct_output.loc[_, "days_to_death"] = None

In [25]:
df_rct_output

,*type,project_id,*submitter_id,*cases.submitter_id,age_at_index,cause_of_death_primary,cause_of_death_secondary,cur_employ_stat,days_to_death,death_related_to,education,ethnicity,gender,marital,race,sex,vital_status,year_of_birth,year_of_death
0,demographic,ARDaC-AlcHepNet,11055_clinical_demographic,11055_clinical,35.1,NaN,NaN,Yes,NaN,NaN,Completed High School,Non-hispanic,Male,Divorced,White,Male,Alive,1985,None
1,demographic,ARDaC-AlcHepNet,11058_clinical_demographic,11058_clinical,29.1,Decompensated liver disease,Acute respiratory failure Aspiration pneumonia...,No,48,NaN,Completed High School,Hispanic or Latino,Female,"Single, never married",White,Female,Dead,1991,2021
2,demographic,ARDaC-AlcHepNet,11066_clinical_demographic,11066_clinical,26.5,NaN,NaN,No,NaN,NaN,Trade School/Some College,Hispanic or Latino,Male,"Single, never married",Unknown,Male,Alive,1994,None
3,demographic,ARDaC-AlcHepNet,11067_clinical_demographic,11067_clinical,48.2,Anoxic brain injury secondary to pulseless ele...,Pulseless electrical activity,No,62,NaN,Standard College/University,Non-hispanic,Male,Divorced,White,Male,Dead,1972,2021
4,demographic,ARDaC-AlcHepNet,11071_clinical_demographic,11071_clinical,43.1,Cardiac arrest in setting of decompensated cir...,Acute respiratory failure Renal failure,No,17,NaN,Completed High School,Non-hispanic,Male,Married,White,Male,Dead,1978,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,demographic,ARDaC-AlcHepNet,81077_clinical_demographic,81077_clinical,31.9,NaN,NaN,No,NaN,NaN,Trade School/Some College,Non-hispanic,Male,"Single, never married",White,Male,Alive,1989,None
143,demographic,ARDaC-AlcHepNet,81080_clinical_demographic,81080_clinical,39.8,NaN,NaN,Yes,NaN,NaN,Completed Graduate/Professional Program,Non-hispanic,Male,Divorced,White,Male,Alive,1981,None
144,demographic,ARDaC-AlcHepNet,81091_clinical_demographic,81091_clinical,56.3,NaN,NaN,Yes,NaN,NaN,Completed Graduate/Professional Program,Non-hispanic,Male,Divorced,White,Male,Alive,1965,None
145,demographic,ARDaC-AlcHepNet,81096_clinical_demographic,81096_clinical,41.2,NaN,NaN,Yes,NaN,NaN,Completed High School,Non-hispanic,Male,Married,White,Male,Alive,1980,None


In [26]:
# Export the "DEMOGRAPHIC" node for Clinical Trial study.
rct_output_path = "demographic_rct_DCC_data_release_v2-0-0.tsv"
df_rct_output.to_csv(rct_output_path, sep="\t", index=False, header=True)
print(f"Clinical Trial patients file saved as: {rct_output_path}")

Clinical Trial patients file saved as: demographic_rct_DCC_data_release_v2-0-0.tsv
